In [4]:
!pip install faster-whisper flask pyngrok gdown -q

from flask import Flask, request, jsonify
from faster_whisper import WhisperModel
from pyngrok import ngrok
import requests
import os
import gdown
import re

# ==========================================
# NGROK AUTH TOKEN (PUT YOUR TOKEN HERE)
# ==========================================
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

# ==========================================
# CREATE APP
# ==========================================
app = Flask(__name__)

# Load Whisper model (fast + good quality)
model = WhisperModel("base", compute_type="int8")

print("✅ Whisper model loaded")

# ==========================================
# HELPER: Convert Google Drive links
# ==========================================
def convert_drive_link(url):
    """
    Converts Google Drive share links into direct download links
    """
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if match:
        file_id = match.group(1)
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    return url

# ==========================================
# TRANSCRIBE ENDPOINT
# ==========================================
@app.route("/transcribe", methods=["POST"])
def transcribe():

    try:
        data = request.json
        audio_url = data.get("audio_url")

        if not audio_url:
            return jsonify({"error": "Missing audio_url"}), 400

        print("\n⬇ Downloading:", audio_url)

        # Convert Drive links automatically
        audio_url = convert_drive_link(audio_url)

        # Download audio
        if "drive.google.com" in audio_url:
            gdown.download(audio_url, "audio.mp3", quiet=False)
        else:
            r = requests.get(audio_url)
            with open("audio.mp3", "wb") as f:
                f.write(r.content)

        # Debug file size
        size = os.path.getsize("audio.mp3")
        print("📁 File size:", size, "bytes")

        if size < 5000:
            return jsonify({
                "error": "Downloaded file too small — probably not audio."
            }), 400

        # TRANSCRIBE
        print("🎙 Transcribing...")
        segments, info = model.transcribe("audio.mp3")

        text = ""
        for segment in segments:
            text += segment.text + " "

        text = text.strip()

        print("✅ Transcript:", text[:150])

        # Cleanup
        os.remove("audio.mp3")

        return jsonify({"transcript": text})

    except Exception as e:
        print("❌ ERROR:", str(e))
        return jsonify({"error": str(e)}), 500



public_url = ngrok.connect(5000)
print("🌍 PUBLIC URL:", public_url)


app.run(port=5000)


✅ Whisper model loaded
🌍 PUBLIC URL: NgrokTunnel: "https://maisie-unpresumed-cozily.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



⬇ Downloading: https://drive.google.com/uc?export=download&id=https://drive.google.com/file/d/1v-zhAw_2KZhqoGttYb-5w8LanfCnSlqW


Downloading...
From: https://drive.google.com/uc?export=download&id=1v-zhAw_2KZhqoGttYb-5w8LanfCnSlqW
To: /content/audio.mp3
100%|██████████| 1.24M/1.24M [00:00<00:00, 46.6MB/s]


📁 File size: 1242529 bytes
🎙 Transcribing...


...

✅ Transcript: Hey team, I've been thinking about something important this week.  A lot of founders believe they have a marketing problem,  but honestly most of the 

⬇ Downloading: https://drive.google.com/uc?export=download&id=https://drive.google.com/file/d/1AQ-7n2qShul69hHvabcehKtG1B8_ulMo


Downloading...
From: https://drive.google.com/uc?export=download&id=1AQ-7n2qShul69hHvabcehKtG1B8_ulMo
To: /content/audio.mp3
100%|██████████| 1.24M/1.24M [00:00<00:00, 47.7MB/s]


📁 File size: 1242529 bytes
🎙 Transcribing...


INFO:werkzeug:127.0.0.1 - - [14/Feb/2026 17:09:44] "POST /transcribe HTTP/1.1" 200 -


✅ Transcript: Hey team, I've been thinking about something important this week.  A lot of founders believe they have a marketing problem,  but honestly most of the 
